In [1]:

from backend.data.db_utils import *

In [2]:
db = connect_to_db()

Connected to PostgreSQL at localhost:5454, db 'products_db'


In [3]:
# new groups table schema:
"""
CREATE EXTENSION IF NOT EXISTS "uuid-ossp";
CREATE TABLE groups (
    id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
    name TEXT NOT NULL,
    main_category TEXT,
    sub_category TEXT,
    name_embedding VECTOR(768),
    clean_name TEXT
);

"""

main_category = 'Млечни производи'
sub_category = 'Млеко'
db = connect_to_db()
query = f"SELECT * FROM products WHERE sub_category = '{sub_category}'"
cursor = db.cursor(cursor_factory=RealDictCursor)
cursor.execute(query)
products = cursor.fetchall()
for product in products:
    print(product['name'], product['market'])

    """
    SELECT
    *,
    1 - (name_embedding <=> (SELECT name_embedding FROM products WHERE name = 'Alpsko – Млеко без лактоза 3.5%mm 1l')) as similarity
    FROM products
    WHERE sub_category = 'Млеко'
        AND name_embedding IS NOT NULL
    ORDER BY similarity desc
    """

Connected to PostgreSQL at localhost:5454, db 'products_db'
Млеко &#8216;Zbregov 1л 2.8% zito
Млеко Битолско 1л 1.5% zito
Alpi – Млеко 3,2% 1l reptil
Alpsko – Млеко 1.5% 1l reptil
Alpsko – Млеко 1l reptil
Alpsko – Млеко без лактоза 1.5% 1l reptil
Alpsko – Млеко без лактоза 3.5%mm 1l reptil
Alpsko – Млеко протеин 1l reptil
Bimilk – Битолско Млеко 1.5% 1l reptil
Bimilk – Битолско УХТ млеко 3.2% 1l reptil
Bimilk – Млеко 10 витамини 2.8% 1l reptil
Bimilk – Млеко 2,8% 1l reptil
Bimilk – Битолско млеко +Д3 витамин 3.2% 1l reptil
Bimilk light-Млеко 0.9%1l reptil
Bimilk Млеко  0,5l reptil
GreenFlo Barista Gold – Млеко за кафе 3.8%мм 1l reptil
Lavazza Classico – Млеко за кафе 250g reptil
Milkos – Млеко 2.8% 1l reptil
Milkos – Млеко 3.2% 1l reptil
Milram Kafee Sahne – Млеко за кафе 14g reptil
Pekabesko – Пекабела млеко 3.2% 1l reptil
Pekabesko – Пекабела млеко 1.6% 1l reptil
Zbregov – Млеко 1% 1l reptil
Zbregov – Млеко 2,8%мм 1l reptil
Zbregov – Млеко 2.8% 2l reptil
Zbregov Vindija – Млеко 0.9% 

In [57]:
from time import strftime

# # migrate all products from {market}_products to a new "products" table
# """
# new products table schema:
# id - uuid
# group_id - fk to groups table
# name - text
# description (categories) - text
# price - integer
# singular_price - text
# in_stock - boolean
# image - text
# link - text
# main_category - text
# sub_category - text
# reasoning - float (0-1)
# market - text
# name_embedding - vector(768)
# ETL_loadtime - timestamp
# categorized_at - timestamp
# last_updated - timestamp
# """
# table products is not created yet, so we will create it first and index on id, name and name_embedding columns
create_table_query = """CREATE TABLE IF NOT EXISTS products (
    id UUID PRIMARY KEY,
    group_id UUID DEFAULT NULL,
    name TEXT,
    description TEXT,
    price INTEGER,
    singular_price TEXT,
    in_stock BOOLEAN,
    image TEXT DEFAULT NULL,
    link TEXT DEFAULT NULL,
    main_category TEXT DEFAULT NULL,
    sub_category TEXT DEFAULT NULL,
    confidence FLOAT DEFAULT NULL,
    reasoning TEXT DEFAULT NULL,
    market TEXT,
    name_embedding VECTOR(768) DEFAULT NULL,
    ETL_loadtime TIMESTAMP,
    categorized_at TIMESTAMP DEFAULT NULL,
    last_updated TIMESTAMP
)"""

cur.execute(create_table_query)
create_index_query = """CREATE INDEX IF NOT EXISTS idx_products_name ON products (name)"""
cur.execute(create_index_query)
create_index_query = """CREATE INDEX IF NOT EXISTS idx_products_name_embedding ON products USING hnsw (name_embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64)"""
cur.execute(create_index_query)
conn.commit()
# markets = ['kam', 'ramstore', 'reptil', 'stokomak', 'vero', 'zito']
markets = ['zito']
BATCH_SIZE = 5000
for market in markets:
    query = f"""
SELECT
    kp.id,
    kp.name,
    kp.price,
    kp.singular_price,
    kp.in_stock,
    pc.description,
    pc.main_category,
    pc.sub_category,
    pc.confidence,
    pc.reasoning,
    pc.market,
    pc.categorized_at,
    pc.name_embedding
--     kp.image
FROM {market}_products kp
JOIN products_categorized pc ON kp.id = pc.id
"""
    cur.execute(query)
    products = cur.fetchall()

    batch_rows = []
    for product in products:  # limit to first 20 products for testing
        _id = product[0]
        name = product[1]
        price = product[2]
        singular_price = product[3]
        in_stock = True if product[4] == 1 else False
        description = product[5]
        main_category = product[6]
        sub_category = product[7]
        confidence_raw = product[8]
        try:
            confidence = float(confidence_raw) if confidence_raw is not None else None
        except (TypeError, ValueError):
            confidence = None
        reasoning = product[9]
        categorized_at = product[11]
        name_embedding = product[12]
        # image = product[13]
        ETL_loadtime = strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        last_updated = strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        # print(reasoning, description, confidence, sep="\n", end='\n\n')
        batch_rows.append((
            _id, name, description, price, singular_price, in_stock,
            main_category, sub_category, confidence, reasoning, market,
            categorized_at, name_embedding, ETL_loadtime, last_updated, image
        ))

        if len(batch_rows) >= BATCH_SIZE:
            insert_query = """INSERT INTO products (id, name, description, price, singular_price, in_stock, main_category, sub_category, confidence, reasoning, market, categorized_at, name_embedding, ETL_loadtime, last_updated, image) VALUES %s"""
            execute_values(cur, insert_query, batch_rows)
            conn.commit()
            batch_rows.clear()

    if batch_rows:
        insert_query = """INSERT INTO products (id, name, description, price, singular_price, in_stock, main_category, sub_category, confidence, reasoning, market, categorized_at, name_embedding, ETL_loadtime, last_updated, image) VALUES %s"""
        execute_values(cur, insert_query, batch_rows)
        conn.commit()
